# 085 — Espacios latentes y autoencoders variacionales

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**VAE** (Kingma y Welling, 2013): modelo de variable latente z ~ N(0, I),
x ~ p_θ(x|z), entrenado maximizando la **cota inferior de la evidencia**:

```text
log p_θ(x) ≥ ELBO = E_{q_φ(z|x)}[log p_θ(x|z)] − KL(q_φ(z|x) ‖ p(z))
```

El **encoder** q_φ(z|x) = N(μ_φ(x), σ_φ²(x)) aproxima la posterior intratable;
la brecha de la cota es exactamente KL(q_φ(z|x) ‖ p_θ(z|x)).

**Truco de reparametrización**: z = μ + σ ⊙ ε con ε ~ N(0, I) convierte el
muestreo en una función diferenciable de φ y permite retropropagar.

**KL en forma cerrada** (gaussianas 1D, prior N(0,1)):
`KL = −log σ_q + (σ_q² + μ_q²)/2 − 1/2`.

## 🧮 Ejemplo de referencia

Con μ_q = 0.5, σ_q = 0.8, decoder g(z) = 2z (varianza 1) y observación x = 2.0:

- KL = −log 0.8 + (0.64 + 0.25)/2 − 0.5 = **0.1681**
- ε = 0.5 → z = 0.5 + 0.8·0.5 = 0.9; g(z) = 1.8
- log p(x|z) = −½log(2π) − (2.0 − 1.8)²/2 = **−0.9389**
- ELBO ≈ −0.9389 − 0.1681 = **−1.1071**, es decir log p(x) ≥ −1.1071 en esperanza.

Verifícalo a mano antes de ejecutar el laboratorio.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("generation", seed=85)
show(result)


## Reflexión

1. Si el término KL de todas las dimensiones latentes cae a ≈0 durante el entrenamiento, ¿qué le pasó al modelo (posterior collapse) y por qué las reconstrucciones pueden seguir siendo buenas?
2. ¿Por qué sin el truco de reparametrización no fluye el gradiente del término de reconstrucción hacia los parámetros φ del encoder, y por qué REINFORCE sería una alternativa peor?
3. ¿Qué diferencia hay entre interpolar en el espacio latente de un VAE y en el de un autoencoder determinista, y qué término del ELBO explica esa diferencia?